<a href="https://colab.research.google.com/github/shashankshekhar28/face-mask-detection-system/blob/main/face_mask_detection_system_vgg16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install tensorflow

In [20]:
!pip install opencv-python

In [21]:
import cv2

In [22]:
import os
from tensorflow.keras.preprocessing import image


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/mask detection system.zip"
extract_path = "/content/drive/MyDrive/mask_detection"

# Create folder
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")

Extraction complete!


In [5]:
!ls /content/drive/MyDrive/mask_detection

'mask detection system'


In [6]:
categories = ['with_mask', 'without_mask']
data = []

base_path = "/content/drive/MyDrive/mask_detection/mask detection system/train"

for category in categories:
    path = os.path.join(base_path, category)
    label = categories.index(category)

    for file in os.listdir(path):
        img_path = os.path.join(path, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))

        data.append([img, label])

print("Data loaded:", len(data))

Data loaded: 1376


In [7]:
len(data)

1376

In [8]:
import random
random.shuffle(data)

In [9]:
x=[]
y=[]
for features,label in data:
    x.append(features)
    y.append(label)

In [10]:
len(x)

1376

In [11]:
len(y)

1376

In [12]:
import numpy as np
x=np.array(x)
y=np.array(y)

In [13]:
x.shape

(1376, 224, 224, 3)

In [14]:
y.shape

(1376,)

In [15]:
X=x/255 #standardization

In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [17]:
X_train.shape

(1100, 224, 224, 3)

In [18]:
X_test.shape

(276, 224, 224, 3)

In [24]:
from keras.applications.vgg16 import VGG16
vgg=VGG16()
vgg.summary()

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
from keras import Sequential
model=Sequential()


In [26]:
for layer in vgg.layers[:-1]:
    model.add(layer)

In [27]:
for layer in model.layers:
    layer.trainable=False
model.summary()    #it freezes trainable parameters

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 134,260,544 (512.16 MB)

In [28]:
from keras.layers import Dense
model.add(Dense(1,activation='sigmoid'))
model.summary()          #4097 total trainable params with 4096 from previous layer and 1 bias of new layer

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         4,097 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,264,641 (512.18 MB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 134,260,544 (512.16 MB)

In [29]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [30]:
model.fit(X_train,y_train,epochs=8,validation_data=(X_test,y_test))

Epoch 1/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 38s 703ms/step - accuracy: 0.6109 - loss: 0.7008 - val_accuracy: 0.8225 - val_loss: 0.4676
Epoch 2/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 209ms/step - accuracy: 0.9091 - loss: 0.3798 - val_accuracy: 0.9239 - val_loss: 0.3159
Epoch 3/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 210ms/step - accuracy: 0.9245 - loss: 0.2958 - val_accuracy: 0.9058 - val_loss: 0.2907
Epoch 4/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 215ms/step - accuracy: 0.9364 - loss: 0.2425 - val_accuracy: 0.9348 - val_loss: 0.2143
Epoch 5/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 217ms/step - accuracy: 0.9473 - loss: 0.2061 - val_accuracy: 0.9493 - val_loss: 0.1829
Epoch 6/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 223ms/step - accuracy: 0.9509 - loss: 0.1834 - val_accuracy: 0.9529 - val_loss: 0.1730
Epoch 7/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 228ms/step - accuracy: 0.9536 - loss: 0.1639 - val_accuracy: 0.9638 - val_loss: 0.1484
Epoch 8/8
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 232ms/step - accuracy: 0.9636 - loss: 0.1468 - val_accuracy: 0.9674 

In [31]:
from tensorflow.keras.applications.vgg16 import preprocess_input
import numpy as np

def detect_face_mask(img):
    img = cv2.resize(img, (224,224))
    img = img.astype("float32")
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    prob = model.predict(img, verbose=0)[0][0]
    return 1 if prob >= 0.5 else 0



In [ ]:
#video capture part and mask detection
cap=cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    # call the detection method
    img = cv2.resize(frame, (224, 224))
    y_pred = detect_face_mask(img)
    print("Prediction (0/1):", y_pred)

    if y_pred==0:
        draw_label(frame,'Mask',(30,30),(0,255,0))
    else:
        draw_label(frame,'No Mask',(30,30),(0,0,255))

    cv2.imshow('mask detector', frame)
    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cv2.destroyAllWindows()


In [ ]:
#why preprocessing was done
#Because VGG16 was trained on preprocessed ImageNet images, and without preprocessing your input looked “wrong” to the model.
#VGG16 was trained on BGR images (OpenCV style), not RGB.
#VGG16 subtracts these means:

"""B = 103.939
G = 116.779
R = 123.68


Why?

Centers pixel values around 0

Makes learning stable

Matches training distribution

Without this:

Model receives values in 0–255

But it learned on normalized values

Activations become meaningless"""

In [32]:
def draw_label(img,text,pos,bg_color):
    text_size=cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,1,cv2.FILLED)

    end_x=pos[0]+text_size[0][0]+2
    end_y=pos[1]+text_size[0][1]-2
    cv2.rectangle(img,pos,(end_x,end_y),bg_color,cv2.FILLED)
    cv2.putText(img,text,pos,cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,0),1,cv2.LINE_AA)


In [37]:
#experiment with frame count
"""cap=cv2.VideoCapture(0)
#setting frame resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 200)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 200)

cap.set(cv2.CAP_PROP_FPS, 30)

frame_count = 0
label = 0

while True:
    ret, frame = cap.read()
    frame_count += 1

    if frame_count % 5 == 0:
        img = cv2.resize(frame, (224,224))
        label = detect_face_mask(img)

    draw_label(frame, "No mask" if label else "Mask", (30,30),
               (0,255,0) if label else (0,0,255))

    #fps count
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    h, w, _ = frame.shape
    cv2.putText(
        frame,
        f"FPS: {int(fps)}",
        (w - 180, 40),               # RIGHT side
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),               # Yellow
        2
    )
    # ----------------------------------

    cv2.imshow("mask detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break
cv2.destroyAllWindows()"""

'cap=cv2.VideoCapture(0)\n#setting frame resolution\ncap.set(cv2.CAP_PROP_FRAME_WIDTH, 200)\ncap.set(cv2.CAP_PROP_FRAME_HEIGHT, 200)\n\ncap.set(cv2.CAP_PROP_FPS, 30)\n\nframe_count = 0\nlabel = 0\n\nwhile True:\n    ret, frame = cap.read()\n    frame_count += 1\n\n    if frame_count % 5 == 0:\n        img = cv2.resize(frame, (224,224))\n        label = detect_face_mask(img)\n\n    draw_label(frame, "No mask" if label else "Mask", (30,30),\n               (0,255,0) if label else (0,0,255))\n\n    #fps count\n    curr_time = time.time()\n    fps = 1 / (curr_time - prev_time)\n    prev_time = curr_time\n\n    h, w, _ = frame.shape\n    cv2.putText(\n        frame,\n        f"FPS: {int(fps)}",\n        (w - 180, 40),               # RIGHT side\n        cv2.FONT_HERSHEY_SIMPLEX,\n        1,\n        (255, 255, 0),               # Yellow\n        2\n    )\n    # ----------------------------------\n\n    cv2.imshow("mask detector", frame)\n\n    if cv2.waitKey(1) & 0xFF == ord(\'x\'):\n      

In [33]:
haar = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
def detect_face(img):
   coods=haar.detectMultiScale(img) # gives four number sorrunding our face(coordinates)
   print(coods)
   return coods

In [ ]:
#face detection code seperate
cap=cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    coods=detect_face(cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY))

    for x,y,w,h in coods:
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    cv2.imshow('mask detector', frame)
    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cv2.destroyAllWindows()


In [ ]:
#mask detection with face detection
cap=cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    coods=detect_face(cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)) #works only after converting to grayscale

    for x,y,w,h in coods:
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    # call the detection method
    img = cv2.resize(frame, (224, 224))
    y_pred = detect_face_mask(img)
    print("Prediction (0/1):", y_pred)

    if y_pred==0:
        draw_label(frame,'Mask',(30,30),(0,255,0))
    else:
        draw_label(frame,'No Mask',(30,30),(0,0,255))

    cv2.imshow('mask detector', frame)
    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cv2.destroyAllWindows()


In [36]:
#final executable code
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detect_face(gray)   # returns () if no face

    for (x, y, w, h) in faces:
        # CROP FACE
        face = frame[y:y+h, x:x+w]

        # MASK PREDICTION ON FACE ONLY
        img = cv2.resize(face, (224, 224))
        y_pred = detect_face_mask(img)

        # DRAW FACE BOX
        color = (0,255,0) if y_pred == 0 else (0,0,255)
        cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)

        if y_pred==0:
            draw_label(frame,'Mask',(30,30),(0,255,0))
        else:
            draw_label(frame,'No Mask',(30,30),(0,0,255))

    cv2.imshow("mask detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break
cv2.destroyAllWindows()


In [35]:
# Save model in native Keras format (recommended)
model.save("mask_detector_model.keras")

print("Model saved successfully!")


Model saved successfully!
